In [1]:
from sampo.scheduler.genetic import GeneticScheduler
from sampo.schemas.graph import WorkGraph
from LLMHeuristicScheduler.base import LLMHeuristicScheduler
from sampo_api import contractor


from sampo.base import SAMPO

import seaborn as sns
import logging
import re
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict
import os

Can not find native module; switching to default


# Genetic Algrotihm basic SAMPO  vs Genetic Algorithm with LLMs generated init population

Цель исследования:    
    - Проверить сходимость за сколько был достигнут GAP = 0,   
    Гипотеза: Iter_LLM <  Iter_GA    
    - Оптимальность  
    Гипотеза: makespan_LLM < makespan_GA


In [2]:
class StatsCollector:
    def __init__(self,):
        self.items = []

    def add(self, fitness):
        self.items.append(fitness)
    def clear(self):
        self.items = []
class StatsHandler(logging.Handler):
    pattern = re.compile(
        r"-- Generation (?P<generation>\d+), population=(?P<population>\d+), best fitness=\((?P<fitness>\d+)\.0,\) --"
    )
    def __init__(self, collector):
        super().__init__()
        self.collector = collector  # сюда кладём внешний объект
    def emit(self, record):
        msg = self.format(record)
        m = self.pattern.search(msg)
        if not m:
            return
        #generation = int(m.group("generation"))
        fitness = float(m.group("fitness"))
        self.collector.add(fitness)

In [ ]:
def init_experiment(GA_params, model_names, imprortance):
    solvers_dict = {}
    solvers_dict['genetic'] = GeneticScheduler(**GA_params)
    for model_name in model_names:
        model = LLMHeuristicScheduler(model_name, **GA_params, 
                                      imprortance=imprortance)
        solvers_dict[model_name] = model
    return solvers_dict

def experiment_df(experiment_logs):
    res = []
    for name, runs in experiment_logs.items():
        for i, run in enumerate(runs, start=1):
            df = pd.DataFrame({'Makespan' : run, 'Time' : range(1, len(run) + 1), 'run' : i, 'Algorithm' : name})
            res.append(df)
    df_res = pd.concat(res)
    return df_res

def run_experiment(solvers_dict, wg, contractors, N_runs):
    sc = StatsCollector()
    stats_handler = StatsHandler(sc)
    SAMPO.logger.addHandler(stats_handler)
    experiment_logs = defaultdict(list)
    for i in range(N_runs):
        for solver, model in solvers_dict.items():
            model.schedule(wg, contractors)
            experiment_logs[solver].append(sc.items)
            sc.clear()
    return experiment_logs



GA  = {
    'number_of_generation' : 50,
    'size_of_population' : 75,
    'mutate_order' : 0.05,
    'mutate_resources': 0.05,
    'mutate_zones': 0.05} 


#wg , contractors  = WorkGraph.loadf('wgs/100', '100_0'), contractor(N=5)


wg , contractors  = WorkGraph.loadf('wgs/small_synth', 'wg_9'), contractor(N=5)
solvers_dict = init_experiment(GA, ['deepseek_chat', 'deepseek_reasoner'], 1)
# experiment_logs = run_experiment(solvers_dict, wg, contractors, 3)


In [ ]:
# sns.lineplot(experiment_df(experiment_logs),
#               x='Time', y='Makespan', hue='Algorithm', 
#               units='run', estimator=None, alpha=0.7)

# plt.ylim((25, 150))
# plt.xlim((0, 50))

# Проверка гипотезы о весах в ГА

In [ ]:
def tail_share(weights, tail_indices, p_init=1/3):
    W = sum(weights)
    W_tail = sum(weights[tail_indices:])
    return p_init * W_tail / W  # доля в общей популяции


print('Deepseek Chat')
print( tail_share([2, 2, 2, 2, 1, 1, 1] + [0.5] * 9, tail_indices=7), tail_share([2, 2, 2, 2, 1, 1, 1] + [0.5] * 9, tail_indices=7) / 0.33)
print( tail_share([2, 2, 2, 2, 1, 1, 1] + [2] * 9, tail_indices=7), tail_share([2, 2, 2, 2, 1, 1, 1] + [1] * 9, tail_indices=7) / 0.33 )
print( tail_share([2, 2, 2, 2, 1, 1, 1] + [6] * 9, tail_indices=7), tail_share([2, 2, 2, 2, 1, 1, 1] + [5] * 9, tail_indices=7) / 0.33)
print( tail_share([2, 2, 2, 2, 1, 1, 1] + [12] * 9, tail_indices=7), tail_share([2, 2, 2, 2, 1, 1, 1] + [10] * 9, tail_indices=7) / 0.33)


print('Reasoner')
print( tail_share([2, 2, 2, 2, 1, 1, 1] + [0.5] * 5, tail_indices=7) / 0.33)
print( tail_share([2, 2, 2, 2, 1, 1, 1] + [2] * 5, tail_indices=7) / 0.33 )
print( tail_share([2, 2, 2, 2, 1, 1, 1] + [6] * 5, tail_indices=7) / 0.33)
print( tail_share([2, 2, 2, 2, 1, 1, 1] + [12] * 5, tail_indices=7) )


In [ ]:
from sampo.schemas.graph import WorkGraph

def init_experiment(GA_params, model_names, imprt):
    solvers_dict = {}
    solvers_dict['genetic'] = GeneticScheduler(**GA_params)
    for model_name in model_names:
        model = LLMHeuristicScheduler(model_name, **GA_params, imprt=imprt)
        solvers_dict[model_name] = model
    return solvers_dict

def experiment_df(experiment_logs):
    res = []
    for name, runs in experiment_logs.items():
        if name == 'weight_param':
            continue
        for i, run in enumerate(runs, start=1):
            df = pd.DataFrame({'Makespan' : run, 'Time' : range(1, len(run) + 1), 'run' : i, 'Algorithm' : name})
            res.append(df)
    df_res = pd.concat(res)
    df_res['weight_param'] = experiment_logs['weight_param'] 
    return df_res

def run_experiments(GA, wg, weights, contractors, N_runs):
    sc = StatsCollector()
    stats_handler = StatsHandler(sc)
    SAMPO.logger.addHandler(stats_handler)
    res_dfs = []
    for weight in weights:
        experiment_logs = defaultdict(list)
        solvers_dict = init_experiment(GA, ['deepseek_chat', 'deepseek_reasoner'], imprt=weight)
        for _ in range(N_runs):
            for solver, model in solvers_dict.items():
                if weight >= weights[0] and solver == 'genetic':
                # Skip other experiments
                    continue
                model.schedule(wg, contractors)
                experiment_logs[solver].append(sc.items)
                experiment_logs['weight_param'] = weight
                sc.clear()
            res_dfs.append(experiment_df(experiment_logs))
    
    return res_dfs


GA  = {
    'number_of_generation' : 150,
    'size_of_population' : 50,
    'mutate_order' : 0.05,
    'mutate_resources': 0.05,
    'mutate_zones': 0.05} 


weights = (8, 9, 10)


wg , contractors  = WorkGraph.loadf('wgs/small_synth', 'wg_9'), contractor(N=5)
dfs = run_experiments(GA, wg, contractors, 5)


In [ ]:
data = pd.concat(dfs)
data.to_csv('weight_experiment2_df.csv', index=False)

#data.to_csv('weight_experiment2_weights_df.csv', index=False)


# Meta

# GA  = {
#     'number_of_generation' : 150,
#     'size_of_population' : 50,
#     'mutate_order' : 0.05,
#     'mutate_resources': 0.05,
#     'mutate_zones': 0.05} 


#df_experiment.to_csv('weight_experiment_df.csv', index=False)


# Meta

# GA  = {
#     'number_of_generation' : 100,
#     'size_of_population' : 50,
#     'mutate_order' : 0.05,
#     'mutate_resources': 0.05,
#     'mutate_zones': 0.05} 



In [ ]:
# from sampo.schemas.graph import WorkGraph
# import os
# from sampo_api import contractor

# maxx, minn = -float('inf'), float('inf')
# for f in os.listdir('wgs/small_synth'):
#         wg , contractors = WorkGraph.loadf('wgs/small_synth', f[:-5]), contractor(N=5)
#         N = len(wg.nodes) 
#         print( f, N)
#         maxx = max(maxx, N)
#         minn = min(minn, N)

# print(maxx, minn)

# Experiment with structure in GA

In [3]:
import random as rand

rand.sample(['a', 'b', 'c'], k=4, counts=[3, 2, 1])

['b', 'c', 'b', 'a']

In [4]:
import math
math.ceil(0.3)

1

In [27]:
from LLMHeuristicScheduler.base import LLMHeuristicScheduler
from sampo_api import contractor
from sampo.schemas.graph import WorkGraph

wg , contractors  = WorkGraph.loadf('wgs/small_synth', 'wg_9'), contractor(N=5)


GA  = {
    'number_of_generation' : 150,
    'size_of_population' : 50,
    'mutate_order' : 0.05,
    'mutate_resources': 0.05,
    'mutate_zones': 0.05} 

scheduler = LLMHeuristicScheduler('deepseek_reasoner', 
                                  **GA,  
                                 type_init_pop_structure='onlyGeneratedHeurisitcs')

In [28]:
scheduler.schedule(wg, contractors)


[SAMPO] [INFO] Toolbox initialization & first population took 14.466047286987305 ms


5 50 [10, 10, 10, 10, 10]
Genetic optimizing took 13.90385627746582 ms


[SAMPO] [INFO] First population evaluation took 384.5031261444092 ms
[SAMPO] [INFO] -- Generation 1, population=50, best fitness=(68.0,) --
[SAMPO] [INFO] -- Generation 2, population=50, best fitness=(68.0,) --
[SAMPO] [INFO] -- Generation 3, population=50, best fitness=(68.0,) --
[SAMPO] [INFO] -- Generation 4, population=50, best fitness=(68.0,) --
[SAMPO] [INFO] -- Generation 5, population=50, best fitness=(68.0,) --
[SAMPO] [INFO] -- Generation 6, population=50, best fitness=(68.0,) --
[SAMPO] [INFO] -- Generation 7, population=50, best fitness=(68.0,) --
[SAMPO] [INFO] -- Generation 8, population=50, best fitness=(68.0,) --
[SAMPO] [INFO] -- Generation 9, population=50, best fitness=(67.0,) --
[SAMPO] [INFO] -- Generation 10, population=50, best fitness=(67.0,) --
[SAMPO] [INFO] -- Generation 11, population=50, best fitness=(67.0,) --
[SAMPO] [INFO] -- Generation 12, population=50, best fitness=(67.0,) --
[SAMPO] [INFO] -- Generation 13, population=50, best fitness=(67.0,) --
[SAM